# 03 · Building a field bank for breast MRI

The full pipeline on 3D data with the `breast_mri` site profile:

1. **Preprocess** every subject (reorient to RAS, N4, resample, percentile normalisation).
2. **Build a field bank**: register each subject A to `k` partners B with similar breast volume
   (`reg_aladin`, then `reg_f3d -vel`, masked to the breasts), store the SVFs and their QC.
3. **Augment** with `DriftAugment`: each call picks a partner, draws a time per breast (and the
   background time for lesions), and warps image and labels with the same diffeomorphism.

**Data.** Point `DATA_DIR` at a folder with `images/<id>.nii.gz` and `labels/<id>.nii.gz`, one integer
label map per subject following the profile's convention (0 background, 1 breast, 2 fibroglandular tissue,
3 vessels, 4 lesion). For Duke-Breast-Cancer-MRI that means converting the pre-contrast series to NIfTI (e.g.
with `dcm2niix`) and merging the Lew et al. 2024 breast / dense-tissue / vessel masks into one label map;
that conversion is not part of DRIFT. If `DATA_DIR` doesn't exist, the notebook generates small synthetic
breast-like phantoms instead, so it runs end to end anywhere.

Requires the `io` and `viz` extras and NiftyReg on `PATH` (or `NIFTYREG_BIN` set).

In [ ]:
import dataclasses
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from drift.augment import DriftAugment
from drift.bank import FieldBank
from drift.filters import gaussian_smooth
from drift.grid import identity
from drift.io import load_nifti, save_nifti
from drift.jacobian import det
from drift.preprocess import preprocess
from drift.profile import load_profile
from drift.space import ImageSpace

local = Path("..") / "third_party" / "niftyreg" / "install" / "bin"
if "NIFTYREG_BIN" not in os.environ and local.is_dir():
    os.environ["NIFTYREG_BIN"] = str(local.resolve())

DATA_DIR = Path(os.environ.get("DRIFT_BREAST_DIR", "../data/duke"))
WORK = Path("../data/work_03")  # data/ is gitignored
MAX_SUBJECTS = 8  # registration is the slow part: N subjects x K partners runs
K = 3  # partners per subject for this demo (the profile's default is 8)
JOBS = 4  # NiftyReg registrations in parallel (each gets cpu_count // JOBS threads)

## Subjects

Real data if `DATA_DIR` exists, otherwise synthetic phantoms. A phantom is a prone bilateral breast MRI in
caricature, on a RAS grid: a chest-wall slab, two half-ellipsoid breasts of random size (fat, bright on
T1), irregular fibroglandular tissue inside them and, in every other subject, a small lesion.

In [ ]:
def phantom(seed, shape=(96, 64, 48), spacing=2.0):
    """Axis 0 left->right, 1 posterior->anterior, 2 inferior->superior."""
    g = torch.Generator().manual_seed(seed)

    def r(lo, hi):
        return float(lo + (hi - lo) * torch.rand(1, generator=g))

    x = identity(shape)[0]
    wall = 18 + r(-2, 2)
    breasts = torch.zeros(shape, dtype=torch.bool)
    fgt = torch.zeros(shape, dtype=torch.bool)
    for cx in (shape[0] * 0.3 + r(-3, 3), shape[0] * 0.7 + r(-3, 3)):
        rad = torch.tensor([r(12, 18), r(18, 34), r(10, 16)]).view(3, 1, 1, 1)
        c = torch.tensor([cx, wall, shape[2] / 2 + r(-3, 3)]).view(3, 1, 1, 1)
        rel = (x - c) / rad
        b = (rel.norm(dim=0) <= 1) & (x[1] >= wall)
        noise = gaussian_smooth(torch.randn(1, 1, *shape, generator=g), 2.5)[0, 0]
        inner = ((x - c - torch.tensor([0.0, 4.0, 0.0]).view(3, 1, 1, 1)) / (0.6 * rad)).norm(dim=0)
        breasts |= b
        fgt |= b & (inner**2 + 2.5 * noise < 1)
    lab = breasts.long()
    lab[fgt] = 2
    if seed % 2 == 0:
        c = torch.tensor([shape[0] * 0.3, wall + 10, shape[2] / 2]).view(3, 1, 1, 1)
        lab[((x - c).norm(dim=0) < 3.5) & breasts] = 4
    img = torch.zeros(shape)
    img[(x[1] < wall) & (x[1] > 4)] = 0.35  # chest wall
    img[lab == 1], img[lab == 2], img[lab == 4] = 0.9, 0.45, 0.6
    img = gaussian_smooth(img[None, None], 0.7) + 0.02 * torch.randn(1, 1, *shape, generator=g)
    return (
        img.clamp_min(0),
        lab[None, None].to(torch.int16),
        ImageSpace.from_spacing(shape, spacing),
    )


profile = load_profile("breast_mri")
raw = {}
if (DATA_DIR / "images").is_dir():
    for p in sorted((DATA_DIR / "images").glob("*.nii.gz"))[:MAX_SUBJECTS]:
        sid = p.name.removesuffix(".nii.gz")
        lab = DATA_DIR / "labels" / p.name
        raw[sid] = (p, lab if lab.exists() else None)
    # The torch backend is much faster on a GPU; NiftyReg stays the reference.
    backend = "torch" if torch.cuda.is_available() else "niftyreg"
    profile = dataclasses.replace(
        profile,
        registration=dataclasses.replace(profile.registration, backend=backend),
        partners=dataclasses.replace(profile.partners, k=K),
    )
else:
    print(f"{DATA_DIR} not found: using synthetic phantoms")
    (WORK / "raw").mkdir(parents=True, exist_ok=True)
    for i in range(4):
        img, lab, space = phantom(i)
        raw[f"phantom{i}"] = (
            WORK / "raw" / f"phantom{i}.nii.gz",
            WORK / "raw" / f"phantom{i}_lab.nii.gz",
        )
        save_nifti(raw[f"phantom{i}"][0], img, space)
        save_nifti(raw[f"phantom{i}"][1], lab, space)
    # The phantoms are already on a 2 mm RAS grid without a bias field;
    # 4 subjects need fewer partners.
    profile = dataclasses.replace(
        profile,
        preprocess=dataclasses.replace(profile.preprocess, spacing=None, n4=False),
        partners=dataclasses.replace(profile.partners, k=2),
        store_spacing=None,
    )
print(len(raw), "subjects;", profile.preprocess)

## Preprocess

`preprocess` applies the profile's steps to the image and, consistently, to its labels (nearest-neighbour
when resampling). The bank is built on these preprocessed images, and augmentation uses the same grids.

In [ ]:
prep_dir = WORK / "preprocessed"
prep_dir.mkdir(parents=True, exist_ok=True)
prepared = {}
for sid, (img_path, lab_path) in raw.items():
    out_img, out_lab = prep_dir / f"{sid}.nii.gz", prep_dir / f"{sid}_lab.nii.gz"
    if not out_img.exists():
        img, space = load_nifti(img_path)
        lab = None if lab_path is None else load_nifti(lab_path, dtype=torch.int64)[0]
        img, lab, space = preprocess(
            img, space, labels=lab, **dataclasses.asdict(profile.preprocess)
        )
        save_nifti(out_img, img, space)
        if lab is not None:
            save_nifti(out_lab, lab.to(torch.int16), space)
    prepared[sid] = (out_img, out_lab if out_lab.exists() else None)

## Build the bank

`add_subject` writes the registration mask (breast labels dilated by 10 mm, or a smoothed Otsu body mask when
a subject has no labels) and records the breast volume used to pick partners. `build` runs the registrations
and can be interrupted and resumed: pairs already in the bank are skipped.

In [ ]:
bank = FieldBank(WORK / "bank", profile=profile)
for sid, (img, lab) in prepared.items():
    if sid not in bank.subjects:
        bank.add_subject(sid, img, lab)
pairs = bank.plan_pairs()
print(f"{len(pairs)} pairs, e.g. {pairs[:3]}")
bank.build(pairs, jobs=JOBS)
print(bank.summary())

Per-pair QC. `dice` is A's breast mask moved by the field against B's (after affine alignment);
`dice_affine` is the same before the non-rigid step (both need labels). Pairs outside the profile's bounds (folds, extreme
`det J`, low Dice) are excluded from sampling.

In [ ]:
cols = [
    "det_min",
    "det_max",
    "fold_fraction",
    "max_disp_mm",
    "nmi_affine",
    "nmi",
    "dice_affine",
    "dice",
]
print(f"{'pair':<24}" + "".join(f"{c:>14}" for c in cols) + "  passed")
for p in bank.pairs.values():
    vals = "".join(f"{p.qc.get(c, float('nan')):>14.3f}" for c in cols)
    print(f"{p.a + ' <- ' + p.b:<24}{vals}  {p.passed} {'; '.join(p.reasons)}")

## Augment

`DriftAugment` takes its policy, Lagrangian mode and rejection rules from the profile: each breast gets its
own `t ~ U(-0.25, 1.25)`, the background (and any lesion) gets `t_bg ~ U(0, 0.5)`, the times are smoothed
with σ = 6 mm and attached to the tissue (Lagrangian), and samples with `det J` outside `[0.3, 3]` or a
displacement over 40 mm are redrawn. Below: axial slices through one subject, the original first, with the
label outlines (breast, FGT, lesion) and `det J` of each sample. Without labels the regional policy falls back
to one global `t` per sample.

In [ ]:
aug = DriftAugment(
    bank, seed=0, return_field=True, device="cuda" if torch.cuda.is_available() else "cpu"
)
sid = next(s for s in bank.subjects if bank.partners(s))
img, _ = load_nifti(prepared[sid][0])
sample = {"subject": sid, "image": img[0]}
if prepared[sid][1] is not None:
    sample["label"] = load_nifti(prepared[sid][1], dtype=torch.int64)[0][0]
    z = int((sample["label"][0] > 0).nonzero()[:, 2].float().mean())  # through the breasts
else:
    z = img.shape[-1] // 2

samples = [{**sample, "drift": {"partner": "(original)", "times": {}}}]
samples += [aug(sample) for _ in range(5)]
fig, ax = plt.subplots(2, len(samples), figsize=(3.2 * len(samples), 5.6))
for i, smp in enumerate(samples):
    im = smp["image"][0, :, :, z].cpu().T
    ax[0, i].imshow(im, cmap="gray", origin="lower", vmin=0, vmax=1)
    if "label" in smp:
        lb = smp["label"][0, :, :, z].cpu().T
        ax[0, i].contour(
            lb, levels=[0.5, 1.5, 3.5], colors=["y", "c", "r"], linewidths=0.8, origin="lower"
        )
    times = smp["drift"].get("times", {})
    title = smp["drift"]["partner"] + "".join(f"\n{k} {v:+.2f}" for k, v in times.items())
    ax[0, i].set_title(title, fontsize=8)
    if "drift_displacement" in smp:
        dj = det(smp["drift_displacement"][None].cpu())[0, :, :, z].T
        ax[1, i].imshow(dj, cmap="RdBu_r", vmin=0, vmax=2, origin="lower")
        ax[1, i].set_title(f"det J [{float(dj.min()):.2f}, {float(dj.max()):.2f}]", fontsize=8)
    for a in ax[:, i]:
        a.axis("off")
fig.tight_layout()

## Speed

On a GPU the augmentation is cheap enough to run inside the training loop: one exponentiation or path
integration, one Jacobian check and one warp per key. Fields are cached on the device after first use.

In [ ]:
import time

for _ in range(3):
    aug(sample)
if torch.cuda.is_available():
    torch.cuda.synchronize()
t = time.perf_counter()
for _ in range(10):
    aug(sample)
if torch.cuda.is_available():
    torch.cuda.synchronize()
ms = (time.perf_counter() - t) / 10 * 1e3
print(f"{ms:.0f} ms per sample at {tuple(img.shape[2:])} on {aug.device}")